<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

Run date time (IST): 2026-11-25 13:32:27
=================== PORTFOLIO QUALITY ====================
qualified stocks: 84
with latest results: 84
still star stocks: 44


=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.54 C
CAGR/XIRR %:     1.41%
Est FTT Value:   2.74 C
Future Upside:   1.19 C (77.4%)
=================== PERFORMANCE ==========================
Current PnL:             -40.98 L (-21.35%)
Total Booked PnL:        49.23 L (33.02%)
Booked + Current PnL:    8.24 L (5.53%)
CY Booked PnL:           6.36 L (4.12%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               1.18 L (0.77%)
CY Investment:            1.92 C
Reserve:                  2.04 K
=================== P&L DETAIL ============================
Total Profit:             3.04 L
Total Loss:               -44.02 L


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
76,UNITDSPR,1.59,10.30,9.49,20.77,24509.0,24114.0,258258.0,1644.00,-3.42,68.0,X-MC,2.34,64.0,0.98,1.67,24.14,X40N,NTT,BREWERIES
50,M&M,-0.36,8.49,8.28,17.48,27047.0,25575.0,326650.0,3762.88,-3.80,68.0,X-LC,5.23,25.0,0.95,2.12,26.31,X40N,ATH,AUTO
12,BOSCH-HCIL,-0.54,-23.67,30.99,-0.01,30233.0,-30247.0,97558.0,2282.00,763.13,49.0,M-SC,18.18,231.0,-1.00,0.63,66.06,OX40N,BTT,AC
75,TTKPRESTIG,-0.65,-31.65,46.43,0.09,31984.0,-31891.0,68886.0,770.00,-30.80,42.0,M-SC,3.16,250.0,-1.00,0.45,23.99,OX40N,NTT,DURABLES
52,MASFIN,3.28,-19.49,50.62,21.26,39929.0,-19101.0,78879.0,396.02,-37.89,50.0,H-SC,13.78,167.0,-0.48,0.51,7.47,XR,ATH,FINANCE


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
38,HYUNDAI,-1.83,8.17,31.54,42.29,96748.0,23177.0,306748.0,2782.73,26.39,54.0,X-LC,3.15,29.0,0.24,1.99,26.27,X40N,ATH,AUTO
47,JIOFIN,1.39,-21.67,73.73,36.09,231908.0,-87008.0,314537.0,387.00,-30.48,51.0,X-LC,7.04,34.0,-0.38,2.04,8.93,XY24,BTT,FINANCE
44,INFY,1.95,-18.62,75.30,42.66,245644.0,-74634.0,326221.0,1972.00,-40.92,52.0,X-LC,5.71,3.0,-0.30,2.11,16.78,X40,NTT,IT
33,HCLTECH,0.17,-6.49,34.43,25.70,116218.0,-23423.0,337550.0,1837.07,-13.01,51.0,X-LC,9.06,8.0,-0.20,2.19,34.68,X40,ATH,IT
4,ASIANPAINT,1.20,-4.76,31.06,24.83,105439.0,-16948.0,339469.0,3422.40,-16.02,65.0,X-LC,2.77,30.0,-0.16,2.20,24.46,X40,ATH,PAINTS


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
20,DABUR,6.87,-16.63,72.37,43.70,149664.0,-41264.0,206804.0,735.00,-29.32,74.0,X-MC,9.83,70.0,-0.28,1.34,20.28,XY24,BTT,FMCG
64,SHALBY,5.97,-51.00,155.31,25.10,137852.0,-92384.0,88759.0,327.00,NaN,55.0,NaN,20.19,236.0,-0.67,0.58,12.96,XY24,NTT,HEALTHCARE
48,KANSAINER,4.67,-32.49,68.08,13.47,123941.0,-87615.0,182052.0,340.00,-26.54,58.0,H-SC,5.19,162.0,-0.71,1.18,28.68,XY24,NTT,PAINTS
23,DMART,4.48,-11.17,51.53,34.61,177843.0,-43383.0,345126.0,5391.45,-27.19,60.0,X-LC,13.86,14.0,-0.24,2.24,13.53,X40N,ATH,FMCG
36,HINDUNILVR,4.09,-9.58,39.03,25.72,143551.0,-38956.0,367797.0,2922.00,-39.58,69.0,X-LC,8.45,9.0,-0.27,2.38,16.91,XY25,NTT,FMCG


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
69,TANLA,-4.87,-26.84,194.11,115.17,384511.0,-72672.0,198089.0,1903.93,-20.31,52.0,H-SC,26.91,147.0,-0.19,1.28,75.11,AR,ATH,IT
22,DIXON,-3.15,-5.52,45.88,37.82,113045.0,-14406.0,246392.0,18917.17,-55.46,54.0,X-MC,3.68,54.0,-0.13,1.60,34.17,X40N,ATH,IT
68,SYMPHONY,-2.59,-59.57,147.25,-0.03,101888.0,-101947.0,69194.0,1306.00,NaN,47.0,NaN,22.98,205.0,-1.00,0.45,9.23,OX40N,NTT,DURABLES
29,GREENPANEL,-2.58,-59.54,253.78,43.14,187985.0,-109004.0,74074.0,537.00,NaN,55.0,NaN,21.80,225.0,-0.58,0.48,21.92,XY24,NTT,MISC
3,ALKYLAMINE,-2.38,-37.74,209.24,92.54,131532.0,-38101.0,62862.0,4520.85,-47.61,27.0,H-SC,7.78,142.0,-0.29,0.41,19.37,SR,ATH,CHEMICALS


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-1.42,46.60,40.82,106.44,88356.0,68808.0,216452.0,1629.94,-8.75,42.0,M-SC,20.98,220.0,0.78,1.40,64.01,OX40N,ATH,CABLES
65,SIS,-1.31,-16.72,45.62,21.26,42182.0,-18569.0,92463.0,528.00,77.85,37.0,H-SC,4.79,168.0,-0.44,0.60,37.48,OX40N,NTT,MISC
12,BOSCH-HCIL,-0.54,-23.67,30.99,-0.01,30233.0,-30247.0,97558.0,2282.00,763.13,49.0,M-SC,18.18,231.0,-1.00,0.63,66.06,OX40N,BTT,AC
42,INDIGOPNTS,-0.45,-29.90,42.68,0.02,52225.0,-52196.0,122363.0,1408.00,-33.06,50.0,M-SC,6.43,222.0,-1.00,0.79,39.42,OX40N,NTT,PAINTS
75,TTKPRESTIG,-0.65,-31.65,46.43,0.09,31984.0,-31891.0,68886.0,770.00,-30.80,42.0,M-SC,3.16,250.0,-1.00,0.45,23.99,OX40N,NTT,DURABLES


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-1.42,46.6,40.82,106.44,88356.0,68808.0,216452.0,1629.94,-8.75,42.0,M-SC,20.98,220.0,0.78,1.4,64.01,OX40N,ATH,CABLES


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-1.42,46.6,40.82,106.44,88356.0,68808.0,216452.0,1629.94,-8.75,42.0,M-SC,20.98,220.0,0.78,1.40,64.01,OX40N,ATH,CABLES
66,STARHEALTH,0.96,10.2,33.63,47.26,104188.0,28685.0,309808.0,761.00,-19.51,56.0,H-SC,3.25,173.0,0.28,2.01,33.95,XY24,NTT,INSURANCE


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-1.42,46.60,40.82,106.44,88356.0,68808.0,216452.0,1629.94,-8.75,42.0,M-SC,20.98,220.0,0.78,1.40,64.01,OX40N,ATH,CABLES
83,WIPRO,-0.01,-13.93,124.26,93.01,267621.0,-34868.0,215372.0,420.00,-26.02,50.0,M-LC,7.45,98.0,-0.13,1.40,19.45,XR,NTT,IT
69,TANLA,-4.87,-26.84,194.11,115.17,384511.0,-72672.0,198089.0,1903.93,-20.31,52.0,H-SC,26.91,147.0,-0.19,1.28,75.11,AR,ATH,IT
41,INDIAMART,-1.78,-34.70,207.71,100.92,167277.0,-42802.0,80534.0,4675.60,-61.13,41.0,H-SC,16.48,140.0,-0.26,0.52,2.17,AR,ATH,MISC
3,ALKYLAMINE,-2.38,-37.74,209.24,92.54,131532.0,-38101.0,62862.0,4520.85,-47.61,27.0,H-SC,7.78,142.0,-0.29,0.41,19.37,SR,ATH,CHEMICALS


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
54,PAGEIND,1.45,-20.36,66.01,32.21,141093.0,-54634.0,213745.0,50690.45,-39.58,32.0,X-SC,7.49,55.0,-0.39,1.39,2.56,X40,ATH,APPARELS
37,HONAUT,-1.33,-23.98,98.47,50.88,144357.0,-46238.0,146600.0,58192.80,-40.39,36.0,X-SC,9.56,90.0,-0.32,0.95,11.69,X40N,ATH,ELECTRICAL
73,TMPV,0.56,-35.06,130.37,49.60,403723.0,-167197.0,309675.0,600.00,-529.52,40.0,X-MC,17.09,4.0,-0.41,2.01,3.13,XY24,NTT,AUTO
6,AWL,-0.96,-42.77,191.83,67.02,391170.0,-152394.0,203915.0,485.00,-74.77,41.0,X-SC,10.52,81.0,-0.39,1.32,1.43,XY24,NTT,FMCG
10,BATAINDIA,-0.13,-59.63,234.88,35.20,122020.0,-76720.0,51950.0,2096.00,-19.34,45.0,X-SC,6.43,91.0,-0.63,0.34,7.95,X40,NTT,FOOTWEAR


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
1,ABBOTINDIA,0.48,-11.94,31.01,15.37,48841.0,-21355.0,157500.0,34390.26,-29.44,51.0,X-MC,0.15,58.0,-0.44,1.02,6.69,X40,ATH,PHARMA
76,UNITDSPR,1.59,10.30,9.49,20.77,24509.0,24114.0,258258.0,1644.00,-3.42,68.0,X-MC,2.34,64.0,0.98,1.67,24.14,X40N,NTT,BREWERIES
17,CIPLA,2.44,7.80,14.93,23.89,48389.0,23452.0,324107.0,1648.14,41.64,64.0,X-MC,2.66,18.0,0.48,2.10,21.40,X40N,ATH,PHARMA
11,BERGEPAINT,1.31,-13.70,45.21,25.32,100566.0,-35307.0,222442.0,680.00,-22.89,48.0,X-MC,2.69,68.0,-0.35,1.44,16.65,XY24,NTT,PAINTS
4,ASIANPAINT,1.20,-4.76,31.06,24.83,105439.0,-16948.0,339469.0,3422.40,-16.02,65.0,X-LC,2.77,30.0,-0.16,2.20,24.46,X40,ATH,PAINTS


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
6,AWL,-0.96,-42.77,191.83,67.02,391170.0,-152394.0,203915.0,485.00,-74.77,41.0,X-SC,10.52,81.0,-0.39,1.32,1.43,XY24,NTT,FMCG
54,PAGEIND,1.45,-20.36,66.01,32.21,141093.0,-54634.0,213745.0,50690.45,-39.58,32.0,X-SC,7.49,55.0,-0.39,1.39,2.56,X40,ATH,APPARELS
73,TMPV,0.56,-35.06,130.37,49.60,403723.0,-167197.0,309675.0,600.00,-529.52,40.0,X-MC,17.09,4.0,-0.41,2.01,3.13,XY24,NTT,AUTO
26,GILLETTE,0.86,-15.19,51.74,28.69,111409.0,-38571.0,215325.0,10891.01,-39.30,48.0,X-SC,6.22,83.0,-0.35,1.40,4.04,X40,ATH,FMCG
1,ABBOTINDIA,0.48,-11.94,31.01,15.37,48841.0,-21355.0,157500.0,34390.26,-29.44,51.0,X-MC,0.15,58.0,-0.44,1.02,6.69,X40,ATH,PHARMA


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
10,BATAINDIA,-0.13,-59.63,234.88,35.20,122020.0,-76720.0,51950.0,2096.00,-19.34,45.0,X-SC,6.43,91.0,-0.63,0.34,7.95,X40,NTT,FOOTWEAR
55,PGHH,-1.64,-46.89,146.68,31.02,166347.0,-100112.0,113408.0,17484.41,-60.13,52.0,X-SC,21.48,57.0,-0.60,0.74,6.76,X40,ATH,FMCG
37,HONAUT,-1.33,-23.98,98.47,50.88,144357.0,-46238.0,146600.0,58192.80,-40.39,36.0,X-SC,9.56,90.0,-0.32,0.95,11.69,X40N,ATH,ELECTRICAL
2,ACC,0.36,-43.60,209.93,74.81,317490.0,-116892.0,151236.0,3906.00,20.61,60.0,X-SC,10.96,85.0,-0.37,0.98,12.04,XY24,BTT,CEMENT
1,ABBOTINDIA,0.48,-11.94,31.01,15.37,48841.0,-21355.0,157500.0,34390.26,-29.44,51.0,X-MC,0.15,58.0,-0.44,1.02,6.69,X40,ATH,PHARMA


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
46,ITC,2.94,-11.77,45.88,28.70,177130.0,-51522.0,386073.0,452.00,-16.33,78.0,X-LC,4.91,1.0,-0.29,2.50,21.51,X40,NTT,FMCG
71,TCS,1.52,-24.78,68.51,26.75,251513.0,-120957.0,367118.0,4208.34,-39.92,61.0,X-LC,6.26,2.0,-0.48,2.38,27.33,X40,ATH,IT
44,INFY,1.95,-18.62,75.30,42.66,245644.0,-74634.0,326221.0,1972.00,-40.92,52.0,X-LC,5.71,3.0,-0.30,2.11,16.78,X40,NTT,IT
73,TMPV,0.56,-35.06,130.37,49.60,403723.0,-167197.0,309675.0,600.00,-529.52,40.0,X-MC,17.09,4.0,-0.41,2.01,3.13,XY24,NTT,AUTO
33,HCLTECH,0.17,-6.49,34.43,25.70,116218.0,-23423.0,337550.0,1837.07,-13.01,51.0,X-LC,9.06,8.0,-0.20,2.19,34.68,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
43,INDUSINDBK,0.86,-28.29,85.32,32.89,47237.0,-21842.0,55364.0,1800.00,79.23,61.0,L-MC,2.32,258.0,-0.46,0.36,29.29,XR,NTT,BANKS
12,BOSCH-HCIL,-0.54,-23.67,30.99,-0.01,30233.0,-30247.0,97558.0,2282.00,763.13,49.0,M-SC,18.18,231.0,-1.00,0.63,66.06,OX40N,BTT,AC
79,VALIANTORG,1.95,-32.00,156.11,74.15,173669.0,-52357.0,111248.0,838.00,NaN,36.0,NaN,33.74,145.0,-0.30,0.72,65.55,XR,NTT,CHEMICALS
69,TANLA,-4.87,-26.84,194.11,115.17,384511.0,-72672.0,198089.0,1903.93,-20.31,52.0,H-SC,26.91,147.0,-0.19,1.28,75.11,AR,ATH,IT
27,GLAXO,0.14,8.59,24.96,35.69,54013.0,17116.0,216400.0,3380.16,-21.07,57.0,X-MC,5.76,56.0,0.32,1.40,28.61,X40,ATH,PHARMA


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
81,VOLTAS,0.21,-4.47,51.99,45.20,95227.0,-8577.0,183165.0,1856.00,4.20,69.0,X-MC,14.78,79.0,-0.09,1.19,29.05,XY25,NTT,AC
51,MARUTI,0.40,3.51,21.71,25.99,76343.0,11940.0,351650.0,17120.06,6.92,76.0,X-LC,8.19,33.0,0.16,2.28,25.93,XR,ATH,AUTO
34,HDFCBANK,3.01,5.75,27.37,34.70,104677.0,20798.0,382450.0,1073.00,-19.12,70.0,X-LC,4.60,26.0,0.20,2.48,23.85,X40,BTT,BANKS
50,M&M,-0.36,8.49,8.28,17.48,27047.0,25575.0,326650.0,3762.88,-3.80,68.0,X-LC,5.23,25.0,0.95,2.12,26.31,X40N,ATH,AUTO
76,UNITDSPR,1.59,10.30,9.49,20.77,24509.0,24114.0,258258.0,1644.00,-3.42,68.0,X-MC,2.34,64.0,0.98,1.67,24.14,X40N,NTT,BREWERIES


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,23.35
1,25,52.24
2,50,82.80


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    39.62
SC    27.95
MC    21.80
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.48
XY24     22.62
X40N     19.47
XY25     10.30
XR        9.79
OX40N     6.54
AR        3.13
SR        0.68
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    38.22
X-MC    20.38
H-SC    13.82
X-SC     8.67
M-SC     5.46
M-LC     1.40
M-MC     1.06
L-MC     0.36
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,19.02,-20.11,63.43
IT,12.49,-33.06,102.39
FINANCE,10.81,-22.15,74.13
AUTO,9.26,-11.98,53.11
PAINTS,5.61,-22.17,44.11
INSURANCE,5.52,2.94,33.53
MISC,5.45,-46.99,106.06
BANKS,5.28,-10.10,65.32
PHARMA,4.52,2.75,21.67


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,4006346.0,19
X40,2092140.0,16
XR,1593444.0,13
X40N,1432861.0,12
AR,1025620.0,6
OX40N,779505.0,10
XY25,700967.0,6
SR,308846.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,2353566.0,17
H-SC,2312572.0,15
X-MC,1821598.0,14
X-SC,1468891.0,8
M-SC,528776.0,7
M-LC,267621.0,1
M-MC,82061.0,1
L-MC,47237.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,,sum,count
Conviction,Criteria,,
X-LC,X40,1211932.0,8
H-SC,XY24,988166.0,5
X-SC,XY24,708660.0,2
X-MC,XY24,653953.0,3
H-SC,AR,551788.0,2
X-MC,X40N,550279.0,5
X-SC,X40,540869.0,4
X-LC,X40N,492504.0,4
H-SC,XR,391861.0,4


In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 34.0 seconds
